# Predictive analysis: Zambia fertility risk proxies and maternal/child health outcomes

This notebook analyzes World Bank-style national indicator data for Zambia (`health_zmb.csv`). It treats adolescent fertility, total fertility, family-planning satisfaction, and skilled birth attendance as **national aggregate proxies** for fertility and reproductive-health risk contexts. These are not individual-level measures of parity, birth spacing, maternal age at each birth, or household risk.

The workflow supports a Markdown-first report: all tables, figures, code snippets, validation results, and limitations are designed to be carried into `report.md`. PDF generation is intentionally out of scope for this project version.

## Main methodological guardrails

- Interpolation is useful for exploratory trend visualization, but it can overstate predictive performance when sparse survey indicators are expanded to annual values.
- The original leave-one-out cross-validation (LOOCV) model is retained only as a reference because it trains on future years when predicting earlier years.
- The primary validation check below uses expanding-window, time-forward evaluation and compares models against a previous-year naive baseline.
- All claims are descriptive and predictive, not causal.


## 1. Setup


In [1]:
from pathlib import Path
import os
import re
import warnings

os.environ.setdefault("MPLCONFIGDIR", str(Path.cwd() / ".matplotlib-cache"))

import matplotlib
matplotlib.use("Agg")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.base import clone
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import RidgeCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import LeaveOneOut, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore", category=UserWarning)
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 120

def project_root() -> Path:
    cwd = Path.cwd()
    if (cwd / "health_zmb.csv").exists():
        return cwd
    for p in [cwd, *cwd.parents]:
        if (p / "health_zmb.csv").exists():
            return p
    return cwd

BASE = project_root()
DATA_PATH = BASE / "health_zmb.csv"
FIG_DIR = BASE / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)
print("DATA_PATH:", DATA_PATH.resolve())
print("FIG_DIR:", FIG_DIR.resolve())


DATA_PATH: /Users/lwandokasuba/github/lwandokasuba/ds-assignments/CSC6701/report/health_zmb.csv
FIG_DIR: /Users/lwandokasuba/github/lwandokasuba/ds-assignments/CSC6701/report/figures


## 2. Load data and define indicators


In [2]:
raw = pd.read_csv(DATA_PATH)
raw["Year"] = pd.to_numeric(raw["Year"], errors="coerce")
raw["Value"] = pd.to_numeric(raw["Value"], errors="coerce")
raw = raw.dropna(subset=["Year", "Indicator Code", "Value"])
raw["Year"] = raw["Year"].astype(int)

RISK_CODES = {
    "SP.ADO.TFRT": "adolescent_fertility",
    "SP.DYN.TFRT.IN": "total_fertility",
    "SH.FPL.SATM.ZS": "fp_modern_satisfied_pct",
    "SH.STA.BRTC.ZS": "skilled_birth_pct",
}

OUTCOME_CODES = {
    "SH.STA.MMRT": "maternal_mortality",
    "SH.DYN.MORT": "under5_mortality",
    "SP.DYN.IMRT.IN": "infant_mortality",
    "SH.ANM.ALLW.ZS": "anemia_wra_pct",
}

EXTRA_SURVEY = {
    "SH.STA.STNT.FE.ZS": "stunting_female_under5_pct",
}

KEEP_CODES = list(RISK_CODES) + list(OUTCOME_CODES) + list(EXTRA_SURVEY)
rename_map = {**RISK_CODES, **OUTCOME_CODES, **EXTRA_SURVEY}
sub = raw[raw["Indicator Code"].isin(KEEP_CODES)].copy()

print("Rows in source file:", len(raw))
print("Rows retained for selected indicators:", len(sub))
print("Countries:", ", ".join(sorted(raw["Country ISO3"].dropna().unique())))


Rows in source file: 10076
Rows retained for selected indicators: 345
Countries: ZMB


## 3. Indicator coverage and missingness


In [3]:
def markdown_table(df: pd.DataFrame, index: bool = False) -> str:
    if index:
        df = df.reset_index()
    df = df.copy()
    df.columns = [str(c) for c in df.columns]
    rows = [[str(x) for x in df.columns]]
    for _, row in df.iterrows():
        rows.append(["" if pd.isna(x) else str(x) for x in row.tolist()])
    widths = [max(len(row[i]) for row in rows) for i in range(len(rows[0]))]
    def fmt(row):
        return "| " + " | ".join(row[i].ljust(widths[i]) for i in range(len(row))) + " |"
    sep = "| " + " | ".join("-" * widths[i] for i in range(len(widths))) + " |"
    return "\n".join([fmt(rows[0]), sep] + [fmt(r) for r in rows[1:]])

selected_span = pd.RangeIndex(int(sub["Year"].min()), int(sub["Year"].max()) + 1)
coverage_rows = []
for code_id, short_name in rename_map.items():
    s = sub.loc[sub["Indicator Code"].eq(code_id), ["Year", "Value"]].drop_duplicates("Year")
    years = sorted(s["Year"].astype(int).tolist())
    coverage_rows.append({
        "indicator_code": code_id,
        "short_name": short_name,
        "observed_rows": len(years),
        "first_year": min(years) if years else np.nan,
        "last_year": max(years) if years else np.nan,
        "missing_years_in_1960_2024_span": len(selected_span) - len(set(years)),
    })

coverage = pd.DataFrame(coverage_rows)
display(coverage)
print(markdown_table(coverage))


,indicator_code,short_name,observed_rows,first_year,last_year,missing_years_in_1960_2024_span
0,SP.ADO.TFRT,adolescent_fertility,65,1960,2024,0
1,SP.DYN.TFRT.IN,total_fertility,65,1960,2024,0
2,SH.FPL.SATM.ZS,fp_modern_satisfied_pct,6,1992,2018,59
3,SH.STA.BRTC.ZS,skilled_birth_pct,9,1992,2019,56
4,SH.STA.MMRT,maternal_mortality,39,1985,2023,26
5,SH.DYN.MORT,under5_mortality,65,1960,2024,0
6,SP.DYN.IMRT.IN,infant_mortality,65,1960,2024,0
7,SH.ANM.ALLW.ZS,anemia_wra_pct,24,2000,2023,41
8,SH.STA.STNT.FE.ZS,stunting_female_under5_pct,7,1992,2018,58


| indicator_code    | short_name                 | observed_rows | first_year | last_year | missing_years_in_1960_2024_span |
| ----------------- | -------------------------- | ------------- | ---------- | --------- | ------------------------------- |
| SP.ADO.TFRT       | adolescent_fertility       | 65            | 1960       | 2024      | 0                               |
| SP.DYN.TFRT.IN    | total_fertility            | 65            | 1960       | 2024      | 0                               |
| SH.FPL.SATM.ZS    | fp_modern_satisfied_pct    | 6             | 1992       | 2018      | 59                              |
| SH.STA.BRTC.ZS    | skilled_birth_pct          | 9             | 1992       | 2019      | 56                              |
| SH.STA.MMRT       | maternal_mortality         | 39            | 1985       | 2023      | 26                              |
| SH.DYN.MORT       | under5_mortality           | 65            | 1960       | 2024      | 0                         

## 4. Pivot, interpolate for exploratory trend analysis, and show table previews


In [4]:
def pivot_wide(frame: pd.DataFrame, rename: dict[str, str]) -> pd.DataFrame:
    """Long -> wide: one row per year; latest duplicate per (year, code) wins."""
    slim = frame[["Year", "Indicator Code", "Value"]].sort_values("Year")
    slim = slim.drop_duplicates(subset=["Year", "Indicator Code"], keep="last")
    w = slim.pivot(index="Year", columns="Indicator Code", values="Value")
    w = w.rename(columns=rename)
    return w.sort_index()

wide = pivot_wide(sub, rename_map)
year_min, year_max = int(wide.index.min()), int(wide.index.max())
full_idx = pd.RangeIndex(year_min, year_max + 1, name="Year")
wide = wide.reindex(full_idx)

# This interpolation is used for visual trend continuity and for the explicitly labeled
# interpolated reference model. It is not treated as causal evidence.
wide_interp = wide.interpolate(method="linear", limit_direction="both")

preview_cols = [
    "adolescent_fertility",
    "total_fertility",
    "fp_modern_satisfied_pct",
    "skilled_birth_pct",
    "maternal_mortality",
    "under5_mortality",
    "infant_mortality",
]
preview_cols = [c for c in preview_cols if c in wide.columns]
wide_head = wide[preview_cols].head().round(2)
wide_tail = wide[preview_cols].tail().round(2)

print("Wide table head (uninterpolated):")
print(markdown_table(wide_head, index=True))
print("\nWide table tail (uninterpolated):")
print(markdown_table(wide_tail, index=True))
display(wide_head)
display(wide_tail)


Wide table head (uninterpolated):
| Year   | adolescent_fertility | total_fertility | fp_modern_satisfied_pct | skilled_birth_pct | maternal_mortality | under5_mortality | infant_mortality |
| ------ | -------------------- | --------------- | ----------------------- | ----------------- | ------------------ | ---------------- | ---------------- |
| 1960.0 | 179.23               | 6.95            |                         |                   |                    | 205.0            | 99.4             |
| 1961.0 | 180.86               | 6.98            |                         |                   |                    | 201.3            | 98.1             |
| 1962.0 | 182.54               | 7.01            |                         |                   |                    | 197.8            | 96.9             |
| 1963.0 | 185.14               | 7.06            |                         |                   |                    | 194.8            | 96.0             |
| 1964.0 | 185.02       

Indicator Code,adolescent_fertility,total_fertility,fp_modern_satisfied_pct,skilled_birth_pct,maternal_mortality,under5_mortality,infant_mortality
Year,,,,,,,
1960,179.23,6.95,NaN,NaN,NaN,205.0,99.4
1961,180.86,6.98,NaN,NaN,NaN,201.3,98.1
1962,182.54,7.01,NaN,NaN,NaN,197.8,96.9
1963,185.14,7.06,NaN,NaN,NaN,194.8,96.0
1964,185.02,7.08,NaN,NaN,NaN,192.2,95.3


Indicator Code,adolescent_fertility,total_fertility,fp_modern_satisfied_pct,skilled_birth_pct,maternal_mortality,under5_mortality,infant_mortality
Year,,,,,,,
2020,120.71,4.32,NaN,NaN,99.0,52.5,35.7
2021,118.74,4.25,NaN,NaN,140.0,50.4,35.6
2022,117.40,4.18,NaN,NaN,89.0,49.5,35.4
2023,115.91,4.10,NaN,NaN,85.0,49.1,35.0
2024,115.48,4.04,NaN,NaN,NaN,48.4,34.6


## 5. Exploratory visualizations


In [5]:
plot_cols = [
    "adolescent_fertility",
    "maternal_mortality",
    "under5_mortality",
    "fp_modern_satisfied_pct",
]
plot_cols = [c for c in plot_cols if c in wide_interp.columns]

fig, ax = plt.subplots(figsize=(10, 5))
for col in plot_cols:
    ax.plot(wide_interp.index, wide_interp[col], marker="o", ms=3, lw=1.2, label=col)
ax.set_xlabel("Year")
ax.set_ylabel("Value (indicator units)")
ax.set_title("Selected national indicators (linearly interpolated where sparse)")
ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
fig.tight_layout()
fig.savefig(FIG_DIR / "fig01_timeseries_key_indicators.png", bbox_inches="tight")
plt.close(fig)

corr_feats = [
    "adolescent_fertility",
    "total_fertility",
    "fp_modern_satisfied_pct",
    "skilled_birth_pct",
    "maternal_mortality",
    "under5_mortality",
    "infant_mortality",
    "anemia_wra_pct",
]
corr_feats = [c for c in corr_feats if c in wide_interp.columns]
cm = wide_interp[corr_feats].corr()

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt=".2f", cmap="vlag", center=0, ax=ax)
ax.set_title("Correlation matrix (interpolated national series)")
fig.tight_layout()
fig.savefig(FIG_DIR / "fig02_correlation_heatmap.png", bbox_inches="tight")
plt.close(fig)

print("Saved exploratory figures:")
print("- figures/fig01_timeseries_key_indicators.png")
print("- figures/fig02_correlation_heatmap.png")


Saved exploratory figures:
- figures/fig01_timeseries_key_indicators.png
- figures/fig02_correlation_heatmap.png


## 6. Modeling helpers


In [6]:
feature_cols_full = [c for c in RISK_CODES.values() if c in wide_interp.columns]
feature_cols_annual = ["adolescent_fertility", "total_fertility"]
feature_cols_annual = [c for c in feature_cols_annual if c in wide_interp.columns]

def build_lagged_frame(source: pd.DataFrame, target: str, features: list[str]) -> pd.DataFrame:
    X = source[features].shift(1)
    y = source[target]
    dfm = pd.concat([X, y.rename("y")], axis=1).dropna()
    return dfm

def rmse(ya, yp):
    return float(np.sqrt(mean_squared_error(ya, yp)))

def regression_metrics(y, yp):
    y = np.asarray(y, dtype=float)
    yp = np.asarray(yp, dtype=float)
    return {
        "MAE": float(mean_absolute_error(y, yp)),
        "RMSE": rmse(y, yp),
        "R2": float(r2_score(y, yp)) if len(y) > 1 else np.nan,
    }

def make_models():
    ridge = Pipeline([
        ("scaler", StandardScaler()),
        ("model", RidgeCV(alphas=np.logspace(-3, 3, 25))),
    ])
    rf = RandomForestRegressor(
        n_estimators=300,
        max_depth=3,
        random_state=42,
        min_samples_leaf=2,
    )
    return {"Ridge": ridge, "RandomForest": rf}

def expanding_window_predictions(estimator, X: pd.DataFrame, y: pd.Series, initial_train: int = 20):
    preds = []
    actuals = []
    years = []
    if len(y) <= initial_train:
        return pd.DataFrame(columns=["Year", "actual", "predicted"])
    for i in range(initial_train, len(y)):
        model = clone(estimator)
        model.fit(X.iloc[:i], y.iloc[:i])
        preds.append(float(model.predict(X.iloc[[i]])[0]))
        actuals.append(float(y.iloc[i]))
        years.append(int(y.index[i]))
    return pd.DataFrame({"Year": years, "actual": actuals, "predicted": preds}).set_index("Year")

def evaluate_expanding_window(dfm: pd.DataFrame, features: list[str], initial_train: int = 20) -> pd.DataFrame:
    X = dfm[features]
    y = dfm["y"]
    rows = []

    if len(y) <= initial_train:
        return pd.DataFrame(columns=["model", "n_eval", "MAE", "RMSE", "R2"])

    eval_index = y.index[initial_train:]
    naive_pred = dfm["y"].shift(1).loc[eval_index]
    naive_actual = y.loc[eval_index]
    valid = naive_pred.notna()
    naive_metrics = regression_metrics(naive_actual[valid], naive_pred[valid])
    rows.append({"model": "Previous-year naive", "n_eval": int(valid.sum()), **naive_metrics})

    for name, estimator in make_models().items():
        pred_df = expanding_window_predictions(estimator, X, y, initial_train=initial_train)
        m = regression_metrics(pred_df["actual"], pred_df["predicted"])
        rows.append({"model": name, "n_eval": len(pred_df), **m})

    return pd.DataFrame(rows)

def evaluate_loocv_reference(dfm: pd.DataFrame, features: list[str]) -> pd.DataFrame:
    X = dfm[features]
    y = dfm["y"].values
    rows = []
    loo = LeaveOneOut()
    for name, estimator in make_models().items():
        yp = cross_val_predict(estimator, X, y, cv=loo)
        rows.append({"model": name, "n_eval": len(y), **regression_metrics(y, yp)})
    return pd.DataFrame(rows)


## 7. Interpolated full-series model: LOOCV reference and time-forward validation


In [7]:
targets = [
    ("maternal_mortality", "Maternal mortality ratio"),
    ("under5_mortality", "Under-five mortality (per 1,000)"),
    ("infant_mortality", "Infant mortality (per 1,000)"),
]

loocv_rows = []
expanding_rows = []
importances = {}

for tgt, label in targets:
    if tgt not in wide_interp.columns:
        continue
    dfm = build_lagged_frame(wide_interp, tgt, feature_cols_full)
    loocv = evaluate_loocv_reference(dfm, feature_cols_full)
    loocv.insert(0, "outcome", label)
    loocv_rows.append(loocv)

    expanding = evaluate_expanding_window(dfm, feature_cols_full, initial_train=20)
    expanding.insert(0, "outcome", label)
    expanding_rows.append(expanding)

    rf = make_models()["RandomForest"]
    rf.fit(dfm[feature_cols_full], dfm["y"])
    importances[label] = pd.Series(rf.feature_importances_, index=feature_cols_full).sort_values(ascending=False)

    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    ridge_pred = expanding_window_predictions(make_models()["Ridge"], dfm[feature_cols_full], dfm["y"], initial_train=20)
    axes[0].scatter(ridge_pred["actual"], ridge_pred["predicted"], alpha=0.85)
    lo = min(ridge_pred["actual"].min(), ridge_pred["predicted"].min())
    hi = max(ridge_pred["actual"].max(), ridge_pred["predicted"].max())
    axes[0].plot([lo, hi], [lo, hi], ls="--", c="gray")
    axes[0].set_xlabel("Actual")
    axes[0].set_ylabel("Predicted (Ridge expanding-window)")
    axes[0].set_title(f"{label}: Ridge")

    imp = importances[label]
    axes[1].barh(imp.index[::-1], imp.values[::-1], color="steelblue")
    axes[1].set_title("Random Forest importance")
    fig.suptitle(f"{label} — lagged predictors")
    fig.tight_layout()
    safe = re.sub(r"[^a-z0-9]+", "_", label.lower()).strip("_")
    fig.savefig(FIG_DIR / f"fig03_model_{safe}.png", bbox_inches="tight")
    plt.close(fig)

loocv_results = pd.concat(loocv_rows, ignore_index=True).round(3)
expanding_results = pd.concat(expanding_rows, ignore_index=True).round(3)

print("LOOCV reference on interpolated full-series data (not primary validation):")
print(markdown_table(loocv_results))
print("\nExpanding-window validation on interpolated full-series data (primary validation):")
print(markdown_table(expanding_results))
display(loocv_results)
display(expanding_results)


LOOCV reference on interpolated full-series data (not primary validation):
| outcome                          | model        | n_eval | MAE    | RMSE   | R2    |
| -------------------------------- | ------------ | ------ | ------ | ------ | ----- |
| Maternal mortality ratio         | Ridge        | 64     | 27.037 | 35.311 | 0.965 |
| Maternal mortality ratio         | RandomForest | 64     | 14.703 | 23.671 | 0.984 |
| Under-five mortality (per 1,000) | Ridge        | 64     | 8.122  | 9.578  | 0.964 |
| Under-five mortality (per 1,000) | RandomForest | 64     | 6.816  | 9.048  | 0.968 |
| Infant mortality (per 1,000)     | Ridge        | 64     | 3.764  | 4.205  | 0.967 |
| Infant mortality (per 1,000)     | RandomForest | 64     | 2.682  | 3.592  | 0.976 |

Expanding-window validation on interpolated full-series data (primary validation):
| outcome                          | model               | n_eval | MAE    | RMSE   | R2    |
| -------------------------------- | --------------

,outcome,model,n_eval,MAE,RMSE,R2
0,Maternal mortality ratio,Ridge,64,27.037,35.311,0.965
1,Maternal mortality ratio,RandomForest,64,14.703,23.671,0.984
2,"Under-five mortality (per 1,000)",Ridge,64,8.122,9.578,0.964
3,"Under-five mortality (per 1,000)",RandomForest,64,6.816,9.048,0.968
4,"Infant mortality (per 1,000)",Ridge,64,3.764,4.205,0.967
5,"Infant mortality (per 1,000)",RandomForest,64,2.682,3.592,0.976


,outcome,model,n_eval,MAE,RMSE,R2
0,Maternal mortality ratio,Previous-year naive,44,22.614,30.013,0.974
1,Maternal mortality ratio,Ridge,44,39.197,56.341,0.910
2,Maternal mortality ratio,RandomForest,44,47.884,58.771,0.902
3,"Under-five mortality (per 1,000)",Previous-year naive,44,3.602,4.551,0.992
4,"Under-five mortality (per 1,000)",Ridge,44,9.641,12.520,0.939
5,"Under-five mortality (per 1,000)",RandomForest,44,13.125,16.171,0.898
6,"Infant mortality (per 1,000)",Previous-year naive,44,1.986,2.555,0.990
7,"Infant mortality (per 1,000)",Ridge,44,4.535,5.624,0.952
8,"Infant mortality (per 1,000)",RandomForest,44,5.944,7.448,0.915


## 8. Sensitivity checks


In [8]:
# Sensitivity A: remove sparse survey-only predictors and use annual fertility indicators only.
annual_sensitivity_rows = []
for tgt, label in targets:
    if tgt not in wide_interp.columns:
        continue
    dfm = build_lagged_frame(wide_interp, tgt, feature_cols_annual)
    res = evaluate_expanding_window(dfm, feature_cols_annual, initial_train=20)
    res.insert(0, "outcome", label)
    annual_sensitivity_rows.append(res)
annual_sensitivity = pd.concat(annual_sensitivity_rows, ignore_index=True).round(3)

# Sensitivity B: observed sparse-survey-year check. This is descriptive because complete observed
# lagged rows are too few for reliable predictive model validation.
observed_rows = []
for tgt, label in targets:
    if tgt not in wide.columns:
        continue
    dfobs = build_lagged_frame(wide, tgt, feature_cols_full)
    row = {"outcome": label, "complete_observed_lagged_rows": len(dfobs)}
    for feat in feature_cols_full:
        if len(dfobs) >= 3:
            row[f"corr_lagged_{feat}"] = round(float(dfobs[feat].corr(dfobs["y"])), 3)
        else:
            row[f"corr_lagged_{feat}"] = np.nan
    observed_rows.append(row)
observed_survey_check = pd.DataFrame(observed_rows)

print("Sensitivity A: expanding-window validation using annual fertility features only")
print(markdown_table(annual_sensitivity))
print("\nSensitivity B: complete observed lagged rows without interpolation")
print(markdown_table(observed_survey_check))
display(annual_sensitivity)
display(observed_survey_check)


Sensitivity A: expanding-window validation using annual fertility features only
| outcome                          | model               | n_eval | MAE    | RMSE   | R2    |
| -------------------------------- | ------------------- | ------ | ------ | ------ | ----- |
| Maternal mortality ratio         | Previous-year naive | 44     | 22.614 | 30.013 | 0.974 |
| Maternal mortality ratio         | Ridge               | 44     | 45.746 | 55.615 | 0.912 |
| Maternal mortality ratio         | RandomForest        | 44     | 48.561 | 59.524 | 0.899 |
| Under-five mortality (per 1,000) | Previous-year naive | 44     | 3.602  | 4.551  | 0.992 |
| Under-five mortality (per 1,000) | Ridge               | 44     | 22.642 | 27.102 | 0.714 |
| Under-five mortality (per 1,000) | RandomForest        | 44     | 12.428 | 15.311 | 0.909 |
| Infant mortality (per 1,000)     | Previous-year naive | 44     | 1.986  | 2.555  | 0.99  |
| Infant mortality (per 1,000)     | Ridge               | 44     | 10.45 

,outcome,model,n_eval,MAE,RMSE,R2
0,Maternal mortality ratio,Previous-year naive,44,22.614,30.013,0.974
1,Maternal mortality ratio,Ridge,44,45.746,55.615,0.912
2,Maternal mortality ratio,RandomForest,44,48.561,59.524,0.899
3,"Under-five mortality (per 1,000)",Previous-year naive,44,3.602,4.551,0.992
4,"Under-five mortality (per 1,000)",Ridge,44,22.642,27.102,0.714
5,"Under-five mortality (per 1,000)",RandomForest,44,12.428,15.311,0.909
6,"Infant mortality (per 1,000)",Previous-year naive,44,1.986,2.555,0.990
7,"Infant mortality (per 1,000)",Ridge,44,10.450,13.981,0.702
8,"Infant mortality (per 1,000)",RandomForest,44,5.632,6.984,0.926


,outcome,complete_observed_lagged_rows,corr_lagged_adolescent_fertility,corr_lagged_total_fertility,corr_lagged_fp_modern_satisfied_pct,corr_lagged_skilled_birth_pct
0,Maternal mortality ratio,6,0.794,0.918,-0.910,-0.710
1,"Under-five mortality (per 1,000)",6,0.953,0.945,-0.985,-0.663
2,"Infant mortality (per 1,000)",6,0.953,0.943,-0.983,-0.664


## 9. Interpretation notes for the written report


In [9]:
print("Key report notes:")
print("- Treat national indicators as proxies, not direct observations of individual high-risk fertility behavior.")
print("- Use expanding-window validation as the main predictive evidence.")
print("- Interpret LOOCV on interpolated data as an optimistic reference only.")
print("- Observed complete sparse-survey lagged rows are limited, so interpolation dependence must be disclosed.")
print("- Conclusions should remain descriptive/predictive and non-causal.")


Key report notes:
- Treat national indicators as proxies, not direct observations of individual high-risk fertility behavior.
- Use expanding-window validation as the main predictive evidence.
- Interpret LOOCV on interpolated data as an optimistic reference only.
- Observed complete sparse-survey lagged rows are limited, so interpolation dependence must be disclosed.
- Conclusions should remain descriptive/predictive and non-causal.
